# 10.8 · 关键点检测 / 姿态估计 / Keypoint Detection & Pose Estimation

> **课程定位 / Where this fits**
> 第 8 课，**Part 10 · 计算机视觉**。
> Lesson 8, **Part 10 · Computer Vision**.
>
> 检测给"框"、分割给"掩码"，而**关键点检测**要定位物体上一组**特定的点**：人体的关节(肩、肘、腕…)、人脸的特征点(眼角、嘴角…)、手的指尖。把这些点连起来就得到**姿态(pose)**。它用于动作识别、AR 试妆、手势控制、体育分析。本课讲清核心技巧——**热图回归(heatmap regression)**，并用合成数据**亲手训练**一个关键点网络。
> Detection gives boxes, segmentation gives masks; **keypoint detection** locates a set of **specific points**: human joints (shoulder, elbow, wrist…), facial landmarks, fingertips. Connecting them yields a **pose**. Used in action recognition, AR makeup, gesture control, sports analytics. We'll cover the core trick — **heatmap regression** — and train a keypoint net on synthetic data.
>
> 💼 **实战/面试视角**："为什么用热图而不是直接回归坐标 / 怎么从热图得到坐标 / PCK 指标" 是姿态岗常考。
> 💼 **Practical/interview angle:** "why heatmaps over direct coordinate regression / how to decode coordinates / PCK metric" — pose-role questions.

> 📐 **符号约定 / Notation**
> - 关键点(keypoint) —— 物体上一个有语义的点(如"左肘") / a semantic point on the object
> - 热图(heatmap) —— 每个关键点对应一张"该点在各位置的概率/响应图" / per-keypoint response map

> 💡 **面试相关 / Interview-relevant**
> - "热图回归 vs 直接坐标回归"（出镜率 ★★★★，热图更稳/可微/空间信息）
> - "怎么从热图解码出坐标"（★★★，argmax/soft-argmax）
> - "自顶向下 vs 自底向上 多人姿态"（★★★）
> - "PCK 评价指标"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解关键点/姿态任务与输出形式。
   Understand keypoint/pose tasks and outputs.
2. 掌握**热图回归**，理解它为何优于直接回归坐标。
   Master heatmap regression and why it beats direct coordinate regression.
3. 用合成数据**训练**一个关键点网络并可视化。
   Train a keypoint net on synthetic data and visualize.
4. 学会从热图**解码坐标**并用 PCK 评价。
   Decode coordinates from heatmaps and evaluate with PCK.

## 目录 / TOC
1. [关键点与姿态 ⭐](#1)
2. [热图回归：核心技巧 ⭐](#2)
3. [训练一个关键点网络 ⭐](#3)
4. [解码坐标与 PCK + 小结 ⭐](#4)


<a id="1"></a>
## 1. 关键点与姿态 ⭐ / Keypoints & Pose

**关键点检测**的输出是一组坐标：$K$ 个点 $(x_1,y_1),\dots,(x_K,y_K)$，每个点有固定语义（第 1 个永远是"鼻子"，第 2 个永远是"左肩"…）。把人体的关节点按骨架连起来，就是**姿态估计**。
**Keypoint detection** outputs a set of coordinates: $K$ points $(x_1,y_1),\dots,(x_K,y_K)$, each with fixed semantics (point 1 = "nose," point 2 = "left shoulder," …). Connecting human joints by the skeleton gives **pose estimation**.

多人场景有两种策略（面试点）：
Two strategies for multi-person (interview):
- **自顶向下(top-down)**：先检测出每个人(框)，再对每个人单独估关键点。精度高，人多则慢。
  **Top-down:** detect each person (box) first, then estimate keypoints per person. Accurate, slow with many people.
- **自底向上(bottom-up)**：先检测出图中所有关键点，再把它们"分配"给不同的人。人多也快。
  **Bottom-up:** detect all keypoints first, then group them into persons. Fast even with crowds.

下面用合成"小人"演示关键点+骨架。
Let's demo keypoints + skeleton with a synthetic figure.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
sns.set_theme(style="white")

# 合成一个"3 关键点小人"(头/左手/右手) + 骨架 / synthetic 3-keypoint figure + skeleton
kps = [("头", 18, 32), ("左手", 40, 16), ("右手", 40, 48)]   # (名称, y, x)
fig, ax = plt.subplots(figsize=(4.2, 4.2)); ax.set_xlim(0,64); ax.set_ylim(64,0)
for name, y, x in kps:
    ax.plot(x, y, "o", ms=14); ax.text(x+2, y, name, fontsize=10)
ax.plot([kps[0][2], kps[1][2]], [kps[0][1], kps[1][1]], "-", lw=2, color="gray")   # 头-左手骨架 / skeleton
ax.plot([kps[0][2], kps[2][2]], [kps[0][1], kps[2][1]], "-", lw=2, color="gray")   # 头-右手
ax.set_title("关键点+骨架: 输出每个关节的坐标, 连起来=姿态"); plt.tight_layout(); plt.show()
print("关键点检测: 输出 K 个有语义的点坐标; 连成骨架 = 姿态估计")
print("多人: 自顶向下(先检测人再估点, 准但慢) / 自底向上(先找所有点再分组, 快)")


<a id="2"></a>
## 2. 热图回归：核心技巧 ⭐ / Heatmap Regression

怎么让网络输出一个关键点坐标？两种思路：
How to make a net output a keypoint coordinate? Two approaches:
- **直接回归坐标**：网络末端直接输出数字 $(x, y)$。简单，但实践中**精度差、难训**——把"空间位置"压成两个数丢了空间结构，且对小误差不够敏感。
  **Direct coordinate regression:** output numbers $(x,y)$ directly. Simple but **less accurate, harder to train** — collapsing "spatial location" into two numbers discards spatial structure.
- **热图回归(heatmap regression)** —— 现代主流：网络为**每个关键点输出一整张热图**（与图同尺寸），热图上**该关键点真实位置是一个高斯亮斑**，其余接近 0。网络学的是"逐像素的存在概率"，保留了空间信息、对位置敏感、好训练。
  **Heatmap regression** — the modern standard: the net outputs **a full heatmap per keypoint** (same size as image), with a **Gaussian blob at the true location** and ~0 elsewhere. The net learns "per-pixel presence probability," keeping spatial info, location-sensitive, easy to train.

**目标热图怎么造**：在关键点 $(c_x, c_y)$ 处放一个 2D 高斯 $\exp(-\frac{(x-c_x)^2+(y-c_y)^2}{2\sigma^2})$。预测时取热图最亮点(argmax)就是坐标。
**Building the target heatmap:** place a 2D Gaussian at $(c_x, c_y)$. At inference, the heatmap's brightest point (argmax) is the coordinate.


In [ ]:
def gaussian_heatmap(H, W, cy, cx, sigma=2.5):
    """在 (cy,cx) 处生成 2D 高斯热图 / 2D Gaussian heatmap centered at (cy,cx)."""
    yy, xx = np.mgrid[0:H, 0:W]
    return np.exp(-((xx-cx)**2 + (yy-cy)**2) / (2*sigma**2)).astype(np.float32)

# 演示: 一个关键点 → 一张高斯热图 / one keypoint → one Gaussian heatmap
hm = gaussian_heatmap(64, 64, 32, 40)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
axes[0].imshow(hm, cmap="hot"); axes[0].plot(40, 32, "c+", ms=12); axes[0].set_title("目标热图(关键点处一个高斯亮斑)"); axes[0].axis("off")
axes[1].imshow(hm, cmap="hot"); axes[1].set_title("热图 = 逐像素'该点在这里'的响应"); axes[1].axis("off")
# 3 个关键点 → 3 张热图(各一个通道) / 3 keypoints → 3 heatmaps
multi = np.stack([gaussian_heatmap(64,64,y,x) for _,y,x in [("",18,32),("",40,16),("",40,48)]])
axes[2].imshow(multi.max(0), cmap="hot"); axes[2].set_title("3个关键点 = 3张热图(此处叠加显示)"); axes[2].axis("off")
plt.tight_layout(); plt.show()
print("热图回归: 每个关键点输出一张热图, 真实位置是高斯亮斑; 预测取 argmax 得坐标")
print("优于直接回归坐标: 保留空间信息 + 对位置敏感 + 好训练(现代姿态网络几乎都用)")


<a id="3"></a>
## 3. 训练一个关键点网络 ⭐ / Training a Keypoint Net

合成任务：图里有**一个目标关键点**（最亮的点）外加几个**暗干扰点**和背景噪声，网络要输出一张热图、把亮斑**精确定位到目标点**。我们用一个小**编码器-解码器**（类似 10.7），输出 1 张热图，用 **MSE** 拟合到以真实点为中心的高斯目标。
Synthetic task: the image has **one target keypoint** (the brightest blob) plus a few **dim distractors** and background noise; the net outputs a heatmap that **precisely localizes the target**. We use a small **encoder-decoder** (like 10.7) outputting 1 heatmap, trained with **MSE** to a Gaussian centered on the true point.

> 🔎 **为什么这里只做 1 个关键点**：真实姿态有 K 个关节，就用 **K 个输出通道**（每通道一张热图），原理完全一样。但当多个关节**外观相似又左右对称**（如左右手）时，小网络+少量数据**难以分清该把哪个关节放进哪个通道**，PCK 会明显下降——这正是真实多人/多关节姿态需要**更大模型、更多数据、额外结构约束**的原因。为聚焦讲清"热图回归"本身，这里用干净的单关键点任务。
> 🔎 **Why a single keypoint here:** real pose has K joints → use **K output channels** (one heatmap each), exactly the same principle. But when joints are **similar-looking and symmetric** (e.g. left/right hands), a small net + little data **struggles to assign which joint to which channel**, and PCK drops sharply — precisely why real multi-person/multi-joint pose needs **bigger models, more data, and extra structural constraints**. To focus on "heatmap regression" itself, we use a clean single-keypoint task.


In [ ]:
def make_sample(H=64, W=64, rng=None):
    """一个目标关键点(最亮) + 几个暗干扰点 + 噪声 / one target keypoint + dim distractors + noise."""
    rng = rng or np.random
    img = (rng.normal(0, 0.05, (H, W))).astype(np.float32)        # 背景噪声 / background noise
    yy, xx = np.mgrid[0:H, 0:W]
    for _ in range(rng.randint(2, 5)):                            # 2~4 个暗干扰点 / dim distractors
        dy, dx = rng.randint(6, H-6), rng.randint(6, W-6)
        img += 0.4 * np.exp(-((xx-dx)**2+(yy-dy)**2)/(2*2.0**2)).astype(np.float32)
    cy, cx = rng.randint(8, H-8), rng.randint(8, W-8)             # 目标关键点位置 / target keypoint
    img += np.exp(-((xx-cx)**2+(yy-cy)**2)/(2*2.5**2)).astype(np.float32)   # 目标更亮 / brighter target
    img = np.clip(img, 0, 1)
    hm = gaussian_heatmap(H, W, cy, cx)[None]                     # 目标热图 (1,H,W) / target heatmap
    return img.astype(np.float32), hm.astype(np.float32), (cy, cx)

rng = np.random.RandomState(0); N = 500
X = np.zeros((N,1,64,64), np.float32); Yh = np.zeros((N,1,64,64), np.float32); coords_all = []
for i in range(N):
    im, hm, co = make_sample(rng=rng); X[i,0]=im; Yh[i]=hm; coords_all.append(co)
Xt, Yt = torch.tensor(X), torch.tensor(Yh)
Xtr, Ytr, Xte = Xt[:420], Yt[:420], Xt[420:]
coords_te = coords_all[420:]

class KeypointNet(nn.Module):                            # 小编码器-解码器, 输出 1 张热图 / encoder-decoder → 1 heatmap
    def __init__(s, c=16):
        super().__init__()
        cb = lambda i,o: nn.Sequential(nn.Conv2d(i,o,3,padding=1), nn.ReLU(), nn.Conv2d(o,o,3,padding=1), nn.ReLU())
        s.e1=cb(1,c); s.e2=cb(c,c*2); s.pool=nn.MaxPool2d(2); s.bott=cb(c*2,c*4)
        s.up2=nn.ConvTranspose2d(c*4,c*2,2,stride=2); s.d2=cb(c*4,c*2)
        s.up1=nn.ConvTranspose2d(c*2,c,2,stride=2); s.d1=cb(c*2,c)
        s.out=nn.Conv2d(c, 1, 1)                         # 输出 1 个通道(1个关键点) / 1 output channel
    def forward(s, x):
        e1=s.e1(x); e2=s.e2(s.pool(e1)); b=s.bott(s.pool(e2))
        d2=s.d2(torch.cat([s.up2(b),e2],1)); d1=s.d1(torch.cat([s.up1(d2),e1],1))
        return s.out(d1)

from torch.utils.data import TensorDataset, DataLoader
loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=64, shuffle=True)   # 小批量训练更稳 / mini-batches
torch.manual_seed(0); net=KeypointNet(); opt=torch.optim.Adam(net.parameters(),1e-3); mse=nn.MSELoss()
for ep in range(25):
    net.train()
    for xb, yb in loader: opt.zero_grad(); loss=mse(net(xb), yb); loss.backward(); opt.step()
net.eval()
with torch.no_grad(): pred=net(Xte)
print(f"关键点网络训练完成, 最终训练损失(MSE)={loss.item():.4f}")
fig, axes = plt.subplots(2, 5, figsize=(13, 5.2))
for j in range(5):
    axes[0,j].imshow(Xte[j,0], cmap="gray"); axes[0,j].set_title("输入(目标+暗干扰)", fontsize=8); axes[0,j].axis("off")
    axes[1,j].imshow(pred[j,0], cmap="hot"); axes[1,j].set_title("预测热图(亮斑=目标)", fontsize=8); axes[1,j].axis("off")
fig.suptitle("关键点热图回归: 输入图 → 一张热图, 亮斑精确落在目标点(忽略暗干扰)"); plt.tight_layout(); plt.show()
print("网络输出一张热图; 训练让亮斑对准真实目标点, 同时学会忽略暗干扰点")


<a id="4"></a>
## 4. 解码坐标与 PCK + 小结 ⭐ / Decoding Coordinates & PCK

预测出热图后，**取每张热图的最亮点(argmax)** 就是该关键点的预测坐标。下面解码并把预测点画回原图，再用 **PCK** 评价。
After predicting heatmaps, **take each heatmap's argmax** as the predicted coordinate. We decode, overlay predictions, then evaluate with **PCK**.

**PCK(Percentage of Correct Keypoints)**：预测点落在真实点附近（距离小于某阈值，常按物体尺寸归一化）就算"对"，统计正确比例。这是姿态估计的标准指标。
**PCK (Percentage of Correct Keypoints):** a prediction counts as correct if it's within a threshold distance of the truth (often normalized by object size); report the fraction correct. The standard pose metric.


In [ ]:
def decode(heatmaps):
    """每张热图取 argmax → (y,x) 坐标 / argmax of each heatmap → coordinates."""
    out = []
    for hm in heatmaps:                                  # 遍历 K 张热图 / each of K heatmaps
        idx = hm.reshape(-1).argmax().item()             # 最亮像素的扁平索引 / brightest pixel
        out.append((idx // hm.shape[1], idx % hm.shape[1]))   # 转回 (行,列)=(y,x) / back to (y,x)
    return out

# 可视化预测点 vs 真实点 / predicted vs true keypoints
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for j in range(5):
    ax = axes[j]; ax.imshow(Xte[j,0], cmap="gray"); ax.axis("off")
    (py, px) = decode(pred[j])[0]; (ty, tx) = coords_te[j]
    ax.plot(tx, ty, "go", ms=11, mfc="none", mew=2)      # 真实=绿圈 / true = green circle
    ax.plot(px, py, "r+", ms=11)                          # 预测=红十字 / pred = red plus
    ax.set_title("绿圈=真实 红十=预测", fontsize=8)
plt.tight_layout(); plt.show()
# 在整个测试集上算 PCK / PCK over the whole test set
for thresh in [3, 6]:
    correct = sum(np.hypot(*(np.array(decode(pred[j])[0]) - np.array(coords_te[j]))) < thresh
                  for j in range(len(coords_te)))
    print(f"PCK@{thresh}px = {correct/len(coords_te):.2%}  (预测点落在真实点 {thresh} 像素内的比例)")
print("解码: 热图取 argmax = 关键点坐标; PCK: 距离<阈值算对, 姿态标准指标")


```
关键点检测: 输出K个有语义的点坐标; 连骨架=姿态估计
多人: 自顶向下(先检测人再估点,准慢) vs 自底向上(先找点再分组,快)
核心技巧 热图回归: 每关键点输出一张热图, 真实位置是高斯亮斑(优于直接回归坐标)
为什么热图好: 保留空间信息 + 对位置敏感 + 好训练
解码: 每张热图 argmax → 坐标(soft-argmax 可微更精)
评价 PCK: 预测点距真实<阈值算对; 代表模型 Hourglass/HRNet/OpenPose
```

### 💡 面试速查 / Interview cheat-sheet
1. **热图回归 vs 直接回归**: 热图保留空间信息、对位置敏感、好训练。
   Heatmap vs direct: heatmaps keep spatial info, location-sensitive, easier to train.
2. **目标热图**: 关键点处放2D高斯。
   Target heatmap: a 2D Gaussian at the keypoint.
3. **解码**: 热图 argmax → 坐标(soft-argmax 更精且可微)。
   Decode: heatmap argmax → coordinate (soft-argmax for sub-pixel & differentiable).
4. **多人**: 自顶向下 vs 自底向上。
   Multi-person: top-down vs bottom-up.
5. **PCK**: 距离阈值内算正确, 姿态标准指标。
   PCK: correct within a distance threshold, the standard pose metric.

### 下一节 / Next
**10.9 Vision Transformer (ViT)**——CNN 之外的新范式。ViT 把图像切成小块(patch)当作"词"，用 Transformer 的自注意力处理图像。我们会理解 patch embedding、class token、自注意力，并搭一个迷你 ViT。
**10.9 Vision Transformer** — a new paradigm beyond CNNs. ViT cuts an image into patches treated as "words" and processes them with Transformer self-attention. We'll cover patch embedding, class token, self-attention, and build a mini ViT.
